In [6]:
import numpy as np
import scipy.io
from scipy.signal import butter, lfilter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# EEGNet-specific imports
from EEGModels import EEGNet
from tensorflow.keras import utils as np_utils
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import backend as K

In [7]:
chans=64
kernels=1
n_subjects=35
n_start=500
n_end=750
samples=n_end-n_start
n_freqs=10
n_trials=6

data = {}
for i in range(1, 36):
    file_path = f'D://2024//7-9//dissertation//code//S{i}.mat'
    data[f'S{i}'] = scipy.io.loadmat(file_path)
    

In [3]:
# choose all 35 subjects, first 9 channels
# frequencies 8, 8.2, 8.4, 8.6 Hz for classification
# 1s time length, sample points 501~750

# 受试者键是字符串，选择全部 35 个（字典序）
selected_subjects_keys = [f"S{i}" for i in range(1, n_subjects+1)]

# 初始化一个列表来存储选出的数据
selected_data = []

# 遍历选中的受试者键
for key in selected_subjects_keys:
    # 对每一个受试者数据，选择 9 个通道、2~3 秒 250 个时间点、前 4 个频率、所有实验次数
    subject_data = data[key]['data'][:chans, n_start:n_end, :n_freqs, :]
    selected_data.append(subject_data)

# 将列表转换为 NumPy 数组
selected_data = np.array(selected_data)
X0 = selected_data.transpose(0, 3, 4, 1, 2)
X=X0.reshape(n_subjects*n_freqs*n_trials, chans, samples) # (35 * 10 * 6, 9, 500)

y = []

for i in range(n_subjects):
    for j in range(n_freqs):
        for k in range(n_trials):
            y.append(j)
            
y = np_utils.to_categorical(y)


'''Y0 = selected_data.transpose(1, 2, 0, 3, 4)
indices = np.indices(Y0.shape)
y_indices =indices[3]
#print(y_indices)
#print(y_indices.shape)
y0 = np.array(y_indices)
y = y0.reshape(9, 500, 35*4*6)
print(y[3])
print(y[3].shape)'''

'Y0 = selected_data.transpose(1, 2, 0, 3, 4)\nindices = np.indices(Y0.shape)\ny_indices =indices[3]\n#print(y_indices)\n#print(y_indices.shape)\ny0 = np.array(y_indices)\ny = y0.reshape(9, 500, 35*4*6)\nprint(y[3])\nprint(y[3].shape)'

In [4]:
#pre process data: filtering

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs  # Nyquist Frequency
    low = lowcut / nyq
    high = highcut /nyq
    b, a = butter(order, [low, high], btype='band')
    y = lfilter(b, a, data)
    return y

fs = 250  # 采样频率，根据你的数据实际情况设置
lowcut = 7
highcut = 90

# 初始化一个空的数组，用于存储滤波结果
filtered_X = np.zeros_like(X)

# 对每一个维度进行滤波
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        filtered_X[i, j, :] = bandpass_filter(X[i, j, :], lowcut, highcut, fs, order=5)

X=filtered_X

In [5]:
# 划分数据集并随机打乱
X_train, X_test, Y_train,Y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
#X_validate, X_test, Y_validate,Y_test = train_test_split(X_test, Y_test, test_size=0.5, random_state=42, shuffle=True)


############################# EEGNet portion ##################################



# convert labels to one-hot encodings.
#Y_train      = np_utils.to_categorical(Y_train)
#Y_validate   = np_utils.to_categorical(Y_validate)
#Y_test       = np_utils.to_categorical(Y_test)

# convert data to NHWC (trials, channels, samples, kernels) format. Data 
# contains 60 channels and 151 time-points. Set the number of kernels to 1.
X_train      = X_train.reshape(X_train.shape[0], chans, samples, kernels)
#X_validate   = X_validate.reshape(X_validate.shape[0], chans, samples, kernels)
X_test       = X_test.reshape(X_test.shape[0], chans, samples, kernels)
  
print('X_train shape:', X_train.shape)
print('Y_train shape:', Y_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')

# configure the EEGNet-8,2,16 model with kernel length of 32 samples (other 
# model configurations may do better, but this is a good starting point)
model = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, 
               dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

# compile the model and set the optimizers
model.compile(loss='categorical_crossentropy', optimizer='adam', 
              metrics = ['accuracy'])

# count number of parameters in the model
numParams    = model.count_params()    

# set a valid path for your system to record model checkpoints
checkpointer = ModelCheckpoint(filepath='/tmp/checkpoint.keras', verbose=1,
                               save_best_only=True)

###############################################################################
# if the classification task was imbalanced (significantly more trials in one
# class versus the others) you can assign a weight to each class during 
# optimization to balance it out. This data is approximately balanced so we 
# don't need to do this, but is shown here for illustration/completeness. 
###############################################################################

# the syntax is {class_1:weight_1, class_2:weight_2,...}. Here just setting
# the weights all to be 1
#class_weights = {0:1, 1:1, 2:1, 3:1, 4:1}




################################################################################
# fit the model. Due to very small sample sizes this can get
# pretty noisy run-to-run, but most runs should be comparable to xDAWN + 
# Riemannian geometry classification (below)
################################################################################
fittedModel = model.fit(X_train, Y_train, batch_size = 16, epochs = 50, 
                        verbose = 2, validation_split = 0.2)

###############################################################################
# can alternatively used the weights provided in the repo. If so it should get
# you 93% accuracy. Change the WEIGHTS_PATH variable to wherever it is on your
# system.
###############################################################################

# WEIGHTS_PATH = /path/to/EEGNet-8-2-weights.h5 
# model.load_weights(WEIGHTS_PATH)

###############################################################################
# make prediction on test set.
###############################################################################

probs       = model.predict(X_test)
preds       = probs.argmax(axis = -1)  
acc         = np.mean(preds == Y_test.argmax(axis=-1))
print("Classification accuracy: %f " % (acc))

X_train shape: (1680, 64, 500, 1)
Y_train shape: (1680, 10)
1680 train samples
420 test samples


C:\Users\qianqian\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
84/84 - 14s - 165ms/step - accuracy: 0.1004 - loss: 2.3123 - val_accuracy: 0.1190 - val_loss: 2.3031
Epoch 2/50
84/84 - 11s - 129ms/step - accuracy: 0.1168 - loss: 2.2923 - val_accuracy: 0.1637 - val_loss: 2.2939
Epoch 3/50
84/84 - 12s - 140ms/step - accuracy: 0.2106 - loss: 2.2558 - val_accuracy: 0.3839 - val_loss: 2.2435
Epoch 4/50
84/84 - 11s - 129ms/step - accuracy: 0.3311 - loss: 2.1620 - val_accuracy: 0.4256 - val_loss: 2.0429
Epoch 5/50
84/84 - 11s - 133ms/step - accuracy: 0.3832 - loss: 2.0149 - val_accuracy: 0.3661 - val_loss: 1.8882
Epoch 6/50
84/84 - 11s - 133ms/step - accuracy: 0.3943 - loss: 1.8707 - val_accuracy: 0.4821 - val_loss: 1.6905
Epoch 7/50
84/84 - 11s - 130ms/step - accuracy: 0.4420 - loss: 1.6828 - val_accuracy: 0.5476 - val_loss: 1.4337
Epoch 8/50
84/84 - 11s - 132ms/step - accuracy: 0.5015 - loss: 1.5224 - val_accuracy: 0.6339 - val_loss: 1.2400
Epoch 9/50
84/84 - 12s - 146ms/step - accuracy: 0.5558 - loss: 1.3733 - val_accuracy: 0.7351 - val_loss: